In [1]:
from feedback_forensics.data.operations.core import load_ap, save_ap

ap = load_ap("/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann.json")

In [2]:
ap["annotators"]
issue_language_annotator = "9ef9bb2f"


filtered_comparisons = []
for comp in ap["comparisons"]:
    if comp["annotations"].get(issue_language_annotator, {}).get("pref") == "irrelevant":
        filtered_comparisons.append(comp)

print(f"Filtered down to {len(filtered_comparisons)} comparisons")

ap["comparisons"] = filtered_comparisons[:100]

Filtered down to 101 comparisons


In [4]:
# load og ap

og_ap = load_ap("../exp/outputs/2025-08-23_22-30-59_geminiflash25/results/070_annotations_train_ap.json")



In [5]:
principle_annotators = {
    hash: annotator for hash, annotator in og_ap["annotators"].items() if annotator["type"] == "principle"
}
human_principle_annotators_descriptions = [
    annotator["description"].replace("Human:", "").strip() for annotator in ap["annotators"].values() if "human" in annotator["description"].lower()
]

overlapping_annotators = {
    hash: annotator for hash, annotator in og_ap["annotators"].items() if annotator["description"].replace("Select the response that ", "").strip() in human_principle_annotators_descriptions
}

In [6]:
human_principle_annotators_descriptions

['is more verbose',
 'has more structured formatting',
 'makes more confident statements',
 'is more factually correct',
 'more strictly follows the requested output format',
 'is more concise',
 'has a more avoidant tone',
 'refuses to answer the question',
 'ends with a follow-up question',
 'is more polite',
 'ISSUE_LANGUAGE']

In [7]:
overlapping_annotators

{'bf731c7f': {'description': 'Select the response that is more concise',
  'type': 'principle'},
 '11bb8171': {'description': 'Select the response that is more verbose',
  'type': 'principle'},
 '2dbc7bd4': {'description': 'Select the response that has more structured formatting',
  'type': 'principle'},
 'e50b5135': {'description': 'Select the response that ends with a follow-up question',
  'type': 'principle'},
 '1a8abbd6': {'description': 'Select the response that more strictly follows the requested output format',
  'type': 'principle'},
 '2e2d4304': {'description': 'Select the response that is more polite',
  'type': 'principle'},
 'e14e5dbe': {'description': 'Select the response that has a more avoidant tone',
  'type': 'principle'},
 '187a0209': {'description': 'Select the response that is more factually correct',
  'type': 'principle'},
 'cceeacb8': {'description': 'Select the response that refuses to answer the question',
  'type': 'principle'},
 '3bbb1c19': {'description': '

In [8]:
# add annotations (and annotator) to AP

ap["annotators"].update(overlapping_annotators)

def get_comparison(id: str, ap: dict):
    for comparison in ap["comparisons"]:
        if comparison["id"] == id:
            return comparison
    print(f"Comparison with id {id} not found")

num_not_found = 0
for comparison in ap["comparisons"]:
    og_comparison = get_comparison(comparison["id"], og_ap)
    if og_comparison is not None:
        for ann_hash in overlapping_annotators:
            comparison["annotations"][ann_hash] = og_comparison["annotations"][ann_hash]
    else:
        num_not_found += 1
        comparison["annotations"] = {}

save_ap(ap, "/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann_with_overlapping_annotators_oss.json")
print(f"Number of comparisons not found: {num_not_found}")


📜  | INFO | Created annotated pairs format dataset: /Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann_with_overlapping_annotators_oss.json
📜  | INFO | - Dataset contains 100 comparisons
📜  | INFO | - Dataset contains 22 annotators
Number of comparisons not found: 0


In [9]:
save_ap(ap, file_path="/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann_parsed_multi_oss.json")

📜  | INFO | Created annotated pairs format dataset: /Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann_parsed_multi_oss.json
📜  | INFO | - Dataset contains 100 comparisons
📜  | INFO | - Dataset contains 22 annotators


In [10]:
import feedback_forensics as ff
import pathlib

dataset = ff.DatasetHandler()
data_path = pathlib.Path("/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann_parsed_multi_oss.json")
dataset.add_data_from_path(data_path)
df = dataset.first_handler.df

📜  | INFO | AnnotatedPairs format version: 2.0
📜  | INFO | Removing 0 comparisons with empty responses. Fraction affected: 0.00%
📜  | INFO | Created 200 annotations for 45 model annotators with 45 reference models in 0.05 seconds
📜  | INFO | Loaded data from path: /Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann_parsed_multi_oss.json


In [11]:
annotator_metadata = dataset.get_available_annotators()
def get_annotator_key(in_row_name: str) -> str:
    annotator_keys = []
    for annotator_key, metadata in annotator_metadata.items():
        if metadata["annotator_in_row_name"] == in_row_name:
            annotator_keys.append(annotator_key)

    assert len(annotator_keys) == 1, f"Multiple annotator keys found for {in_row_name}: {annotator_keys}"
    return annotator_keys[0]

In [12]:
import sklearn.metrics

relevant_principles = [
 'is more verbose',
 'has more structured formatting',
 'makes more confident statements',
 'is more factually correct',
 'more strictly follows the requested output format',
 'is more concise',
 'has a more avoidant tone',
 'refuses to answer the question',
 'ends with a follow-up question',
 'is more polite',
]

annotator_data = {}
for principle in relevant_principles:
    annotator_data[principle] = {}
    llm_hash = get_annotator_key(principle)
    human_hash = get_annotator_key(f"Human: {principle}")

    llm_data = df[llm_hash]
    human_data = df[human_hash]

    annotator_data[principle]["kappa"] = sklearn.metrics.cohen_kappa_score(
        df[llm_hash].to_numpy(dtype="str"),
        df[human_hash].to_numpy(dtype="str"),
    )
    annotator_data[principle]["agreement"] = sklearn.metrics.accuracy_score(
        df[llm_hash].to_numpy(dtype="str"),
        df[human_hash].to_numpy(dtype="str"),
    )

    relevant_values = ["text_a", "text_b"]

    df[f"{llm_hash}_relevant"] = df[llm_hash].isin(relevant_values)
    df[f"{human_hash}_relevant"] = df[human_hash].isin(relevant_values)

    annotator_data[principle]["agreement_on_relevance"] = sklearn.metrics.accuracy_score(
        df[f"{llm_hash}_relevant"].to_numpy(dtype="str"),
        df[f"{human_hash}_relevant"].to_numpy(dtype="str"),
    )

    relevant_values = ["text_a", "text_b"]
    relevant_df = df[df[llm_hash].isin(relevant_values) & df[human_hash].isin(relevant_values)]

    annotator_data[principle]["kappa_relevant"] = sklearn.metrics.cohen_kappa_score(
        relevant_df[llm_hash].to_numpy(dtype="str"),
        relevant_df[human_hash].to_numpy(dtype="str"),
    )
    annotator_data[principle]["agreement_relevant"] = sklearn.metrics.accuracy_score(
        relevant_df[llm_hash].to_numpy(dtype="str"),
        relevant_df[human_hash].to_numpy(dtype="str"),
    )
    annotator_data[principle]["prop_both_rel"] = len(relevant_df) / len(df)




../env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
../env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
../env/lib/python3.13/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [13]:
len(df)

100

In [14]:
annotator_data

{'is more verbose': {'kappa': np.float64(0.34071729957805896),
  'agreement': 0.5,
  'agreement_on_relevance': 0.69,
  'kappa_relevant': np.float64(0.8512396694214877),
  'agreement_relevant': 0.9259259259259259,
  'prop_both_rel': 0.54},
 'has more structured formatting': {'kappa': np.float64(0.2497069167643612),
  'agreement': 0.36,
  'agreement_on_relevance': 0.68,
  'kappa_relevant': np.float64(0.8901734104046243),
  'agreement_relevant': 0.9473684210526315,
  'prop_both_rel': 0.38},
 'makes more confident statements': {'kappa': np.float64(0.011399172803389424),
  'agreement': 0.02,
  'agreement_on_relevance': 0.77,
  'kappa_relevant': np.float64(1.0),
  'agreement_relevant': 1.0,
  'prop_both_rel': 0.02},
 'is more factually correct': {'kappa': np.float64(0.03953088666464455),
  'agreement': 0.05,
  'agreement_on_relevance': 0.85,
  'kappa_relevant': np.float64(0.36363636363636376),
  'agreement_relevant': 0.7142857142857143,
  'prop_both_rel': 0.07},
 'more strictly follows the r

In [15]:
import numpy as np

ref_annotator_data = {'is more verbose': {'kappa': np.float64(0.2302955665024632),
  'agreement': 0.5,
  'agreement_on_relevance': 0.71,
  'kappa_relevant': np.float64(0.41090478071908343),
  'agreement_relevant': 0.704225352112676,
  'prop_both_rel': 0.71},
 'has more structured formatting': {'kappa': np.float64(0.1899759524110871),
  'agreement': 0.36,
  'agreement_on_relevance': 0.58,
  'kappa_relevant': np.float64(0.5970149253731343),
  'agreement_relevant': 0.8,
  'prop_both_rel': 0.45},
 'makes more confident statements': {'kappa': np.float64(0.02519962667219755),
  'agreement': 0.06,
  'agreement_on_relevance': 0.5,
  'kappa_relevant': np.float64(0.3414634146341463),
  'agreement_relevant': 0.6666666666666666,
  'prop_both_rel': 0.09},
 'is more factually correct': {'kappa': np.float64(0.014879425346331354),
  'agreement': 0.04,
  'agreement_on_relevance': 0.67,
  'kappa_relevant': np.float64(-0.15384615384615374),
  'agreement_relevant': 0.4,
  'prop_both_rel': 0.1},
 'more strictly follows the requested output format': {'kappa': np.float64(0.010101010101010055),
  'agreement': 0.02,
  'agreement_on_relevance': 0.54,
  'kappa_relevant': np.float64(-0.33333333333333326),
  'agreement_relevant': 0.5,
  'prop_both_rel': 0.04},
 'is more concise': {'kappa': np.float64(0.2630124366651312),
  'agreement': 0.52,
  'agreement_on_relevance': 0.71,
  'kappa_relevant': np.float64(0.4657425742574258),
  'agreement_relevant': 0.7323943661971831,
  'prop_both_rel': 0.71},
 'has a more avoidant tone': {'kappa': np.float64(0.03536977491961413),
  'agreement': 0.04,
  'agreement_on_relevance': 0.93,
  'kappa_relevant': np.float64(np.nan),
  'agreement_relevant': 1.0,
  'prop_both_rel': 0.04},
 'refuses to answer the question': {'kappa': np.float64(0.018822587104525335),
  'agreement': 0.02,
  'agreement_on_relevance': 0.95,
  'kappa_relevant': np.float64(np.nan),
  'agreement_relevant': 1.0,
  'prop_both_rel': 0.02},
 'ends with a follow-up question': {'kappa': np.float64(0.07145740815502621),
  'agreement': 0.08,
  'agreement_on_relevance': 0.93,
  'kappa_relevant': np.float64(0.7272727272727273),
  'agreement_relevant': 0.8888888888888888,
  'prop_both_rel': 0.09},
 'is more polite': {'kappa': np.float64(0.04909560723514217),
  'agreement': 0.08,
  'agreement_on_relevance': 0.61,
  'kappa_relevant': np.float64(0.7272727272727273),
  'agreement_relevant': 0.8888888888888888,
  'prop_both_rel': 0.09}}

for principle, data in ref_annotator_data.items():
    for metric, value in data.items():
        #assert str(value) == str(annotator_data[principle][metric]), f"{principle} {metric} {value} != {annotator_data[principle][metric]}"
        pass

In [16]:
annotator_data

{'is more verbose': {'kappa': np.float64(0.34071729957805896),
  'agreement': 0.5,
  'agreement_on_relevance': 0.69,
  'kappa_relevant': np.float64(0.8512396694214877),
  'agreement_relevant': 0.9259259259259259,
  'prop_both_rel': 0.54},
 'has more structured formatting': {'kappa': np.float64(0.2497069167643612),
  'agreement': 0.36,
  'agreement_on_relevance': 0.68,
  'kappa_relevant': np.float64(0.8901734104046243),
  'agreement_relevant': 0.9473684210526315,
  'prop_both_rel': 0.38},
 'makes more confident statements': {'kappa': np.float64(0.011399172803389424),
  'agreement': 0.02,
  'agreement_on_relevance': 0.77,
  'kappa_relevant': np.float64(1.0),
  'agreement_relevant': 1.0,
  'prop_both_rel': 0.02},
 'is more factually correct': {'kappa': np.float64(0.03953088666464455),
  'agreement': 0.05,
  'agreement_on_relevance': 0.85,
  'kappa_relevant': np.float64(0.36363636363636376),
  'agreement_relevant': 0.7142857142857143,
  'prop_both_rel': 0.07},
 'more strictly follows the r

In [17]:
annotator_data

def create_markdown_table(metrics=['kappa', 'agreement'], sort_by=None, reverse=False):
    """Create a compact markdown table with specified metrics as columns."""
    lines = []

    # Header
    header = "| Principle"
    for metric in metrics:
        if metric == 'kappa':
            header += " | κ"
        elif metric == 'agreement':
            header += " | Agr"
        elif metric == 'kappa_relevant':
            header += " | κ_r"
        elif metric == 'agreement_on_relevance':
            header += " | Relevance agreement"
        elif metric == 'agreement_relevant':
            header += " | Choice agreement"
        elif metric == 'prop_both_rel':
            header += " | Prop"
    header += " |"
    lines.append(header)

    # Separator
    sep = "|---"
    for _ in metrics:
        sep += "|---"
    sep += "|"
    lines.append(sep)

    # Sort data if sort_by is specified
    items = list(annotator_data.items())
    if sort_by and sort_by in annotator_data[list(annotator_data.keys())[0]]:
        items.sort(key=lambda x: x[1][sort_by], reverse=reverse)

    # Data rows
    for principle, data in items:
        row = f"| {principle}"
        for metric in metrics:
            val = data[metric]
            if isinstance(val, (int, float)):
                row += f" | {val:.2f}"
            else:
                row += f" | {val}"
        row += " |"
        lines.append(row)

    return "\n".join(lines)

# Example usage
print(create_markdown_table(['agreement_on_relevance', 'agreement_relevant'], sort_by='agreement_relevant', reverse=True))



| Principle | Relevance agreement | Choice agreement |
|---|---|---|
| makes more confident statements | 0.77 | 1.00 |
| more strictly follows the requested output format | 0.95 | 1.00 |
| has a more avoidant tone | 0.95 | 1.00 |
| refuses to answer the question | 0.98 | 1.00 |
| ends with a follow-up question | 0.91 | 1.00 |
| is more polite | 0.76 | 1.00 |
| has more structured formatting | 0.68 | 0.95 |
| is more verbose | 0.69 | 0.93 |
| is more concise | 0.52 | 0.88 |
| is more factually correct | 0.85 | 0.71 |
